In [1]:
%cd ../../..

/home/hoanghu/projects/Food-Waste-Optimization


In [2]:
import joblib
from itertools import permutations

import numpy as np
import pandas as pd
from sklearn.preprocessing import TargetEncoder
from xgboost import XGBRegressor

# Read data, trained models

In [3]:
path = "trained_models/pos/phase_4/xgb_cat_Nov13.json"

reg = XGBRegressor(tree_method="hist", enable_categorical=True)
reg.load_model(path)

In [4]:
path = "trained_models/encoder/phase_4/targetenc_meal_id_Nov13.pkl"
enc_meal_id = joblib.load(path)

In [5]:
path = "data/processed/phase_4/dim_meals.xlsx"

dim_meals = pd.read_excel(path)

dim_meals.drop(columns=['schoolyear', 'is_kela', 'is_new', 'meal_type_2', 'restaurant'], inplace=True)
dim_meals.columns = ['id', 'type', 'mean']

dim_meals.head()

,id,type,mean
0,9017,vegan,140.165854
1,7201,vegan,121.918212
2,9032,vegan,121.918212
3,9102,vegan,121.918212
4,7010,vegetarian,71.000000


# Forecast

In [6]:
map_mealtype = {
    'meat':    1, # 'meat',
    'fish':    2, # 'fish',
    'vegan':    3, # 'vegan',
    'vegetarian':    4, # 'vegetarian',
    'chicken':    5, # 'chicken'
}

In [7]:
# meal_ids = [7614, 9500047, 1307]
# restaurant = "che"
# date_start = pd.to_datetime("2024-12-01")


# meals = pd.DataFrame({
#     'meal_id': meal_ids,
#     'restaurant': restaurant,
#     'date': pd.to_datetime(date_start)
# })

path = "tmp.csv"
meals = pd.read_csv(path, parse_dates=['date'])

meals.head()

,index,meal_id,date,restaurant
0,21,9029,2024-12-06,che
1,51,1038,2024-12-06,che
2,58,785,2024-12-06,che
3,76,9500048,2024-12-06,che
4,79,1235,2024-12-06,che


In [8]:
feat = meals.merge(dim_meals, left_on='meal_id', right_on='id', how='left').copy().drop(columns='id')
feat.head()

,index,meal_id,date,restaurant,type,mean
0,21,9029,2024-12-06,che,meat,205.118959
1,51,1038,2024-12-06,che,chicken,188.193370
2,58,785,2024-12-06,che,meat,205.118959
3,76,9500048,2024-12-06,che,fish,126.857143
4,79,1235,2024-12-06,che,chicken,188.193370


In [9]:
meal_ids = (
    meals
    .groupby(['index', 'date', 'restaurant'])['meal_id']
    .apply(lambda x: list(x))
    .reset_index()
    .rename(columns={'meal_id': 'meal_ids'})
)
feat = feat.merge(meal_ids, on=['index', 'date', 'restaurant'], how='left')
feat.head()

,index,meal_id,date,restaurant,type,mean,meal_ids
0,21,9029,2024-12-06,che,meat,205.118959,"[9029, 5285, 6558, 785]"
1,51,1038,2024-12-06,che,chicken,188.193370,"[1038, 7614, 9018, 6953]"
2,58,785,2024-12-06,che,meat,205.118959,"[785, 14377, 9067, 9037]"
3,76,9500048,2024-12-06,che,fish,126.857143,"[9500048, 1021, 1308, 1038]"
4,79,1235,2024-12-06,che,chicken,188.193370,"[1235, 7595, 9044, 6930]"


In [10]:
# Create columns for other and encode them
THETA = 5
records = []
for r in feat.itertuples():
    ids = set(r.meal_ids)
    ids.remove(r.meal_id)
    
    for idx_tup, tup in enumerate(permutations(ids)):
        tup = [*tup]

        # pad
        if len(tup) < THETA - 1:
            tup.extend([0]*(THETA - 1 - len(tup)))
        
        for idx, m_id in enumerate(tup):
            records.append({
                'index': r.index,
                'date': r.date,
                'restaurant': r.restaurant,
                'meal_id': r.meal_id,
                'meal_type': r.type,
                'pcs_mean': r.mean,
                'idx_tup': idx_tup,
                'other': idx + 1,
                'meal_id_other': m_id,
            })

feat = (
    pd.DataFrame.from_records(records)
    .merge(dim_meals, how='left', left_on='meal_id_other', right_on='id')
    .drop(columns='id')
    .rename(columns={'type': 'meal_type_other', 'mean': 'pcs_mean_other'})
)
feat.head()

,index,date,restaurant,meal_id,meal_type,pcs_mean,idx_tup,other,meal_id_other,meal_type_other,pcs_mean_other
0,21,2024-12-06,che,9029,meat,205.118959,0,1,785,meat,205.118959
1,21,2024-12-06,che,9029,meat,205.118959,0,2,6558,vegan,35.750000
2,21,2024-12-06,che,9029,meat,205.118959,0,3,5285,vegan,80.000000
3,21,2024-12-06,che,9029,meat,205.118959,0,4,0,NaN,NaN
4,21,2024-12-06,che,9029,meat,205.118959,1,1,785,meat,205.118959


In [13]:



encoded = enc_meal_id.transform(feat['meal_id_other'].to_numpy().reshape(-1, 1))
mask = (feat['meal_id_other'] != 0).astype(np.int32)
feat['meal_id_other_enc'] = encoded.squeeze() * mask

feat['meal_type_other'] = feat['meal_type_other'].map(map_mealtype)

feat = feat.fillna(0)   

cols_idx = ['index', 'date', 'restaurant', 'meal_id', 'meal_type', 'pcs_mean', 'idx_tup']
feat = (
    feat
    .pivot(index=cols_idx, columns='other', values=['pcs_mean_other', 'meal_type_other', 'meal_id_other_enc'])
    .reset_index()
)

cols = [
    'index',
    'date',
    'restaurant',
    'meal_id',
    'meal_type',
    'pcs_mean',

    'idx_tup',

    'meal_id_other1_enc',
    'meal_id_other2_enc',
    'meal_id_other3_enc',
    'meal_id_other4_enc',

    'meal_type_other1',
    'meal_type_other2',
    'meal_type_other3',
    'meal_type_other4',

    'pcs_mean_other1',
    'pcs_mean_other2',
    'pcs_mean_other3',
    'pcs_mean_other4',
]
feat.columns = cols

feat.drop(columns='idx_tup', inplace=True)





# Encode main columns
feat['meal_type'] = feat['meal_type'].map(map_mealtype)

def _encode_date_cyclic(t, period_week: int = 7, period_day: int = 31, period_month: int = 12):
    def get_sin_encoding(x, period: int):
        return np.sin(2 * np.pi * x / period)
    def get_cos_encoding(x, period: int):
        return np.cos(2 * np.pi * x / period)  

    return pd.Series({
        'weekday_sin': get_sin_encoding(t.weekday(), period_week),
        'weekday_cos': get_cos_encoding(t.weekday(), period_week),
        'day_sin': get_sin_encoding(t.day, period_day),
        'day_cos': get_cos_encoding(t.day, period_day),
        'month_sin': get_sin_encoding(t.month, period_month),
        'month_cos': get_cos_encoding(t.month, period_month),
    })

datetime_encoded = feat['date'].apply(_encode_date_cyclic)
feat = pd.concat([feat, datetime_encoded], axis=1)

feat['meal_id_enc'] = enc_meal_id.transform(feat['meal_id'].to_numpy().reshape(-1, 1))


feat['restaurant'] = feat['restaurant'].map({
    'che': 1, #'chemicum',
    'phy': 2, #'physicum',
    'exa': 3, #'exactum'
})


# Assign categorical column type
cols_cat = [
    'restaurant',
    'meal_type',
    'meal_type_other1',
    'meal_type_other2',
    'meal_type_other3',
    'meal_type_other4',
]
for col in cols_cat:
    feat[col] = feat[col].astype('category')

# Keep important columns
cols_X = [
    'weekday_sin',
    'weekday_cos',
    'day_sin',
    'day_cos',
    'month_sin',
    'month_cos',

    'restaurant',

    'meal_id_enc',
    'meal_type',

    'pcs_mean',


    'meal_id_other1_enc',
    'meal_id_other2_enc',
    'meal_id_other3_enc',
    'meal_id_other4_enc',

    'meal_type_other1',
    'meal_type_other2',
    'meal_type_other3',
    'meal_type_other4',

    'pcs_mean_other1',
    'pcs_mean_other2',
    'pcs_mean_other3',
    'pcs_mean_other4',
]
X = feat[cols_X]

X.head()

,weekday_sin,weekday_cos,day_sin,day_cos,month_sin,month_cos,restaurant,meal_id_enc,meal_type,pcs_mean,...,meal_id_other3_enc,meal_id_other4_enc,meal_type_other1,meal_type_other2,meal_type_other3,meal_type_other4,pcs_mean_other1,pcs_mean_other2,pcs_mean_other3,pcs_mean_other4
0,0.0,1.0,0.394356,0.918958,-2.449294e-16,1.0,1,202.259765,3,197.25,...,60.454545,0.0,3.0,2.0,2.0,0.0,85.236388,204.375235,59.104837,0.0
1,0.0,1.0,0.394356,0.918958,-2.449294e-16,1.0,1,202.259765,3,197.25,...,205.000000,0.0,3.0,2.0,2.0,0.0,85.236388,59.104837,204.375235,0.0
2,0.0,1.0,0.394356,0.918958,-2.449294e-16,1.0,1,202.259765,3,197.25,...,60.454545,0.0,2.0,3.0,2.0,0.0,204.375235,85.236388,59.104837,0.0
3,0.0,1.0,0.394356,0.918958,-2.449294e-16,1.0,1,202.259765,3,197.25,...,91.500000,0.0,2.0,2.0,3.0,0.0,204.375235,59.104837,85.236388,0.0
4,0.0,1.0,0.394356,0.918958,-2.449294e-16,1.0,1,202.259765,3,197.25,...,205.000000,0.0,2.0,3.0,2.0,0.0,59.104837,85.236388,204.375235,0.0


In [18]:
feat.head()

,index,date,restaurant,meal_id,meal_type,pcs_mean,meal_id_other1_enc,meal_id_other2_enc,meal_id_other3_enc,meal_id_other4_enc,...,pcs_mean_other3,pcs_mean_other4,weekday_sin,weekday_cos,day_sin,day_cos,month_sin,month_cos,meal_id_enc,pcs_pred
0,21,2024-12-02,1,6159,3,197.25,91.500000,205.000000,60.454545,0.0,...,59.104837,0.0,0.0,1.0,0.394356,0.918958,-2.449294e-16,1.0,202.259765,220.604736
1,21,2024-12-02,1,6159,3,197.25,91.500000,60.454545,205.000000,0.0,...,204.375235,0.0,0.0,1.0,0.394356,0.918958,-2.449294e-16,1.0,202.259765,231.486862
2,21,2024-12-02,1,6159,3,197.25,205.000000,91.500000,60.454545,0.0,...,59.104837,0.0,0.0,1.0,0.394356,0.918958,-2.449294e-16,1.0,202.259765,227.155396
3,21,2024-12-02,1,6159,3,197.25,205.000000,60.454545,91.500000,0.0,...,85.236388,0.0,0.0,1.0,0.394356,0.918958,-2.449294e-16,1.0,202.259765,227.786942
4,21,2024-12-02,1,6159,3,197.25,60.454545,91.500000,205.000000,0.0,...,204.375235,0.0,0.0,1.0,0.394356,0.918958,-2.449294e-16,1.0,202.259765,240.978333


In [23]:
feat['pcs_pred'] = reg.predict(X)

out = feat.groupby(['index', 'date', 'restaurant', 'meal_id'], observed=True)['pcs_pred'].mean().reset_index()
out.head()

# feat.head()

,index,date,restaurant,meal_id,pcs_pred
0,21,2024-12-02,1,6159,234.463364
1,21,2024-12-02,1,6495,67.374901
2,21,2024-12-02,1,8987,204.521072
3,21,2024-12-02,1,9064,88.382240
4,21,2024-12-03,1,5559,38.499310


In [25]:
# TODO: HoangLe [Nov-18]: Convert code from this notebook to Py file